# YOLOv5 Hyperparameter Sweep (PyTorch Lightning + W&B)

This notebook runs a W&B sweep to search for better YOLOv5 training hyperparameters.

It reuses the same project data-loading approach as the overfit notebook, but uses a real train/val split for optimization.

Suggested flow:
1. Run setup cells (imports, model, login)
2. Adjust sweep space in the sweep config cell
3. Launch the sweep agent

In [ ]:
# If needed, uncomment and run in the active kernel.
# %pip install -q lightning wandb torchmetrics ultralytics python-dotenv

In [ ]:
from __future__ import annotations

import os
import random
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

import lightning as L
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from torchmetrics.detection.mean_ap import MeanAveragePrecision

PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config
from detection_pipeline import (
    DetectionDownloadConfig,
    ToothDetectionDataset,
    AugmentedToothDetectionDataset,
    build_detection_records,
    build_detection_train_pipeline,
    load_or_download_detection_dataset,
    split_grouped_records,
)

YOLOV5_DIR = PROJECT_ROOT / 'external' / 'yolov5'
YOLOV5_DIR.parent.mkdir(parents=True, exist_ok=True)
if not YOLOV5_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_DIR)],
        check=True,
    )
if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

from models.yolo import Model
from utils.general import intersect_dicts, non_max_suppression
from utils.loss import ComputeLoss

print('Using project root:', PROJECT_ROOT)
print('Using YOLOv5 code from:', YOLOV5_DIR)

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class SweepBaseConfig:
    image_size: int = 640
    batch_size: int = 16
    num_workers: int = 0
    max_epochs: int = 15
    lr: float = 1e-3
    weight_decay: float = 5e-4
    momentum: float = 0.937
    conf_threshold: float = 0.001
    iou_threshold: float = 0.6
    pretrained_weights: str = 'yolov5s.pt'
    force_download: bool = False
    train_subset_size: int | None = 256
    val_subset_size: int | None = 64
    seed: int = 42
    wandb_project: str = 'tooth-detection-yolo-sweep'
    sweep_count: int = 12


base_cfg = SweepBaseConfig()
seed_everything(base_cfg.seed)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
class YoloTargetAdapterDataset(Dataset):
    def __init__(self, base_dataset: Dataset):
        self.base_dataset = base_dataset

    def __len__(self) -> int:
        return len(self.base_dataset)

    def __getitem__(self, index: int):
        image, target = self.base_dataset[index]
        return image, target


def yolo_collate_fn(batch: list[tuple[torch.Tensor, dict[str, Any]]]):
    images = []
    yolo_targets = []
    metric_targets = []

    for i, (img, tgt) in enumerate(batch):
        images.append(img.float())

        boxes_xyxy = tgt['boxes'].float()
        labels_one_based = tgt['labels'].long()

        metric_targets.append({
            'boxes': boxes_xyxy,
            'labels': (labels_one_based - 1).clamp_min(0),
        })

        if boxes_xyxy.numel() == 0:
            continue

        _, h, w = img.shape
        x1, y1, x2, y2 = boxes_xyxy[:, 0], boxes_xyxy[:, 1], boxes_xyxy[:, 2], boxes_xyxy[:, 3]

        cx = ((x1 + x2) * 0.5) / w
        cy = ((y1 + y2) * 0.5) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        cls = (labels_one_based - 1).float().clamp_min(0)
        batch_idx = torch.full((boxes_xyxy.shape[0],), float(i), dtype=torch.float32)

        packed = torch.stack([batch_idx, cls, cx, cy, bw, bh], dim=1)
        yolo_targets.append(packed)

    images = torch.stack(images, dim=0)
    if yolo_targets:
        yolo_targets = torch.cat(yolo_targets, dim=0)
    else:
        yolo_targets = torch.zeros((0, 6), dtype=torch.float32)

    return images, yolo_targets, metric_targets


class SweepDataModule(L.LightningDataModule):
    def __init__(self, cfg: SweepBaseConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None

    def setup(self, stage: str | None = None):
        _, coco_data, image_dirs = load_or_download_detection_dataset(
            DetectionDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download,
        )
        records, _ = build_detection_records(coco_data, image_dirs)

        train_rec, val_rec, _ = split_grouped_records(
            records,
            train_size=getattr(config, 'TRAIN_RATIO', 0.7),
            val_size=getattr(config, 'VAL_RATIO', 0.15),
            test_size=getattr(config, 'TEST_RATIO', 0.15),
            random_state=self.cfg.seed,
        )

        if self.cfg.train_subset_size is not None:
            train_rec = train_rec[: self.cfg.train_subset_size]
        if self.cfg.val_subset_size is not None:
            val_rec = val_rec[: self.cfg.val_subset_size]

        base_train = ToothDetectionDataset(train_rec, image_size=self.cfg.image_size, output_channels=3)
        aug_train = AugmentedToothDetectionDataset(base_train, build_detection_train_pipeline())
        base_val = ToothDetectionDataset(val_rec, image_size=self.cfg.image_size, output_channels=3)

        self.train_ds = YoloTargetAdapterDataset(aug_train)
        self.val_ds = YoloTargetAdapterDataset(base_val)

        print(f'Train/Val sizes: {len(self.train_ds)}, {len(self.val_ds)}')

    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
            'collate_fn': yolo_collate_fn,
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.cfg.batch_size, shuffle=True, drop_last=True, **self._loader_kwargs())

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.cfg.batch_size, shuffle=False, **self._loader_kwargs())

In [ ]:
class LitYOLOv5(L.LightningModule):
    def __init__(self, cfg: SweepBaseConfig, num_classes: int = 32):
        super().__init__()
        self.save_hyperparameters(ignore=['cfg'])

        self.cfg = cfg
        self.num_classes = num_classes
        self.compute_loss = None

        ckpt_path = Path(cfg.pretrained_weights)
        ckpt = torch.load(ckpt_path, map_location='cpu')
        yolo_cfg = ckpt['model'].yaml
        self.model = Model(yolo_cfg, ch=3, nc=num_classes).float()

        pretrained_state = ckpt['model'].float().state_dict()
        compatible_state = intersect_dicts(pretrained_state, self.model.state_dict(), exclude=['anchor'])
        self.model.load_state_dict(compatible_state, strict=False)

        self.model.hyp = {
            'box': 0.05,
            'cls': 0.3,
            'obj': 0.7,
            'cls_pw': 1.0,
            'obj_pw': 1.0,
            'fl_gamma': 0.0,
            'label_smoothing': 0.0,
            'anchor_t': 4.0,
        }

        self.map_metric = MeanAveragePrecision(box_format='xyxy', class_metrics=False)

    def on_train_start(self):
        self.model.to(self.device)
        self.compute_loss = ComputeLoss(self.model)
        self._fix_compute_loss_device()

    def _fix_compute_loss_device(self):
        if self.compute_loss is None:
            return
        self.compute_loss.device = self.device
        for name, value in vars(self.compute_loss).items():
            if torch.is_tensor(value):
                setattr(self.compute_loss, name, value.to(self.device))
            elif isinstance(value, list):
                setattr(self.compute_loss, name, [item.to(self.device) if torch.is_tensor(item) else item for item in value])
            elif isinstance(value, tuple):
                setattr(self.compute_loss, name, tuple(item.to(self.device) if torch.is_tensor(item) else item for item in value))

    def forward(self, x: torch.Tensor):
        return self.model(x)

    def training_step(self, batch, batch_idx: int):
        images, yolo_targets, _ = batch
        yolo_targets = yolo_targets.to(images.device, non_blocking=True)
        if self.compute_loss is None:
            self.compute_loss = ComputeLoss(self.model)
        self._fix_compute_loss_device()
        preds = self.model(images)
        loss, loss_items = self.compute_loss(preds, yolo_targets)
        self.log('train/loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=images.size(0))
        self.log('train/loss_box', loss_items[0], on_step=True, on_epoch=True, batch_size=images.size(0))
        self.log('train/loss_obj', loss_items[1], on_step=True, on_epoch=True, batch_size=images.size(0))
        self.log('train/loss_cls', loss_items[2], on_step=True, on_epoch=True, batch_size=images.size(0))
        return loss

    def validation_step(self, batch, batch_idx: int):
        images, _, metric_targets = batch
        raw_output = self.model(images)
        preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output
        nms_preds = non_max_suppression(
            preds,
            conf_thres=self.cfg.conf_threshold,
            iou_thres=self.cfg.iou_threshold,
            multi_label=False,
            max_det=300,
        )

        metric_preds = []
        for det in nms_preds:
            if det is None or len(det) == 0:
                metric_preds.append({
                    'boxes': torch.zeros((0, 4), device=self.device),
                    'scores': torch.zeros((0,), device=self.device),
                    'labels': torch.zeros((0,), dtype=torch.long, device=self.device),
                })
                continue
            metric_preds.append({'boxes': det[:, :4], 'scores': det[:, 4], 'labels': det[:, 5].long()})

        metric_targets = [
            {'boxes': target['boxes'].to(self.device), 'labels': target['labels'].to(self.device)}
            for target in metric_targets
        ]
        self.map_metric.update(metric_preds, metric_targets)

    def on_validation_epoch_end(self):
        metrics = self.map_metric.compute()
        self.log('val/map', metrics['map'], prog_bar=True)
        self.log('val/map_50', metrics['map_50'], prog_bar=True)
        self.map_metric.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.model.parameters(),
            lr=self.cfg.lr,
            momentum=self.cfg.momentum,
            weight_decay=self.cfg.weight_decay,
            nesterov=True,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.cfg.max_epochs)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1},
        }

In [ ]:
# Download pretrained YOLOv5 weights if missing.
weights_path = Path(base_cfg.pretrained_weights)
if not weights_path.exists():
    import urllib.request

    url = 'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt'
    urllib.request.urlretrieve(url, str(weights_path))

print('Using weights:', weights_path.resolve())

In [ ]:
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_path = PROJECT_ROOT / '.env'
if load_dotenv is not None and env_path.exists():
    load_dotenv(env_path, override=True)

wandb_key = (os.getenv('WANDB_API_KEY') or '').strip().strip('\"').strip("'")
if not wandb_key:
    raise RuntimeError('WANDB_API_KEY not found. Add it to /work/.env or environment before running sweeps.')

import wandb
os.environ['WANDB_API_KEY'] = wandb_key
wandb.login(key=wandb_key, relogin=True)
print('W&B login successful.')

In [ ]:
sweep_config = {
    # Phase 1: random search to find promising regions.
    'method': 'random',
    'metric': {'name': 'val/map', 'goal': 'maximize'},
    'parameters': {
        'lr': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 3e-3},
        'weight_decay': {'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 1e-3},
        'momentum': {'min': 0.85, 'max': 0.98},
        'batch_size': {'values': [8, 16, 24, 32]},
        'max_epochs': {'values': [10, 15, 20]},
    },
}

# Phase 2 template: copy hot ranges from phase 1 and switch to bayesian optimization.
bayes_refine_template = {
    'method': 'bayes',
    'metric': {'name': 'val/map', 'goal': 'maximize'},
    'parameters': {
        # Example placeholders; tighten these after phase 1 analysis.
        'lr': {'distribution': 'log_uniform_values', 'min': 3e-4, 'max': 1.5e-3},
        'weight_decay': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 3e-4},
        'momentum': {'min': 0.90, 'max': 0.97},
        'batch_size': {'values': [16, 24, 32]},
        'max_epochs': {'values': [15, 20]},
    },
}

print('Phase 1 sweep config:')
print(sweep_config)
print('\nPhase 2 bayes template (edit after random results):')
print(bayes_refine_template)

In [ ]:
def train_sweep_run():
    run = wandb.init(project=base_cfg.wandb_project)
    cfg = SweepBaseConfig(
        image_size=base_cfg.image_size,
        batch_size=int(wandb.config.get('batch_size', base_cfg.batch_size)),
        num_workers=base_cfg.num_workers,
        max_epochs=int(wandb.config.get('max_epochs', base_cfg.max_epochs)),
        lr=float(wandb.config.get('lr', base_cfg.lr)),
        weight_decay=float(wandb.config.get('weight_decay', base_cfg.weight_decay)),
        momentum=float(wandb.config.get('momentum', base_cfg.momentum)),
        conf_threshold=base_cfg.conf_threshold,
        iou_threshold=base_cfg.iou_threshold,
        pretrained_weights=base_cfg.pretrained_weights,
        force_download=base_cfg.force_download,
        train_subset_size=base_cfg.train_subset_size,
        val_subset_size=base_cfg.val_subset_size,
        seed=base_cfg.seed,
        wandb_project=base_cfg.wandb_project,
        sweep_count=base_cfg.sweep_count,
    )

    seed_everything(cfg.seed)
    datamodule = SweepDataModule(cfg)
    model = LitYOLOv5(cfg=cfg, num_classes=32)

    wandb_logger = WandbLogger(project=cfg.wandb_project, log_model=False)

    checkpoint_cb = ModelCheckpoint(
        dirpath=str(PROJECT_ROOT / 'output' / 'checkpoints' / 'sweeps'),
        filename='sweep-{epoch:02d}-{val_map:.4f}',
        monitor='val/map',
        mode='max',
        save_top_k=1,
        save_last=False,
    )

    lr_monitor = LearningRateMonitor(logging_interval='epoch')

    trainer = L.Trainer(
        max_epochs=cfg.max_epochs,
        accelerator='auto',
        devices=1,
        precision='16-mixed' if torch.cuda.is_available() else 32,
        logger=wandb_logger,
        callbacks=[checkpoint_cb, lr_monitor],
        log_every_n_steps=5,
        deterministic=True,
        enable_progress_bar=False,
    )

    trainer.fit(model, datamodule=datamodule)
    val_metrics = trainer.validate(model, datamodule=datamodule, verbose=False)
    val_map = float(val_metrics[0].get('val/map', 0.0))
    wandb.log({'sweep/final_val_map': val_map})
    print(f'Run complete. val/map={val_map:.4f}')
    wandb.finish()

In [ ]:
# Creates a new sweep and runs N trials locally.
sweep_id = wandb.sweep(sweep=sweep_config, project=base_cfg.wandb_project)
print('Sweep ID:', sweep_id)
wandb.agent(sweep_id, function=train_sweep_run, count=base_cfg.sweep_count)

## Notes

- Start with a small subset for faster sweep iterations, then rerun a narrower sweep on full data.
- You can stop/restart the sweep agent; W&B keeps completed trials.
- After sweep completes, copy best hyperparameters into your finetune notebook.